# 🟡 Interacting with Neo4j using Python

This notebook is a hands-on, practical guide that demonstrates how to interact with **Neo4j** (a NoSQL **Graph** database) using the Python language, the official `neo4j` driver, and the **Cypher** query language.

## 🛠️ What is Neo4j?
Neo4j stores structured data as **networks of connections**. Instead of tables (SQL) or documents (MongoDB), data is modeled as:
- **Nodes:** Entities (e.g., Person, Company, Product).
- **Relationships:** Directed connections with specific types between nodes (e.g., `AMIGO_DE`, `TRABALHA_NA`, `COMPROU`).
- **Properties:** Key-value pairs associated with both nodes and relationships.

### Conceptual Summary

| Property | Details |
|---|---|
| **Paradigm** | Graph Database |
| **Query Language** | Cypher (declarative, visual, ASCII-pattern-based) |
| **Storage** | Nodes and relationships as first-class citizens (not as join tables) |
| **When to use** | Social networks, recommendations, fraud detection, knowledge graphs, dependency analysis, routing and logistics |
| **When NOT to use** | Simple tabular data, massive aggregation operations (OLAP), data without meaningful relationships |

### Cypher Visual Vocabulary

```
(n)              → Node
(n:Pessoa)       → Node with Label (type) 'Pessoa'
(n {nome: 'X'})  → Node with property
-[r]->           → Directed relationship
-[r:AMIGO_DE]->  → Relationship with type 'AMIGO_DE'
```

### Local Connection Details (Docker Compose):
- **Host:** `localhost`
- **Bolt Port:** `7688` (high-performance binary protocol used by the driver)
- **HTTP Port:** `7475` (Neo4j Browser web interface)
- **Authentication:** None (`NEO4J_AUTH=none` in docker-compose.yml)

## 📋 Prerequisites

Before running this notebook, make sure that:

1. **Docker** is installed and running on your machine.
2. The project containers have been started with `make up` or `docker compose up -d`.
3. The `neo4j` container is running (check with `docker compose ps`).

> **💡 Tip:** You can visually explore the graphs created in this notebook by accessing [http://localhost:7475](http://localhost:7475) in your browser.

## 2. Connecting to the Database
Let's import the `GraphDatabase` class and create a driver instance pointing to the **Bolt** port (Neo4j's optimized binary protocol).

> **💡 Key Concept:** The **Bolt** protocol (port 7687) is the high-performance communication channel between the Python driver and Neo4j. The HTTP port (7475) is used only by the browser web interface.

> **Note:** Since we disabled authentication in docker-compose (`NEO4J_AUTH=none`), we pass `auth=None`.

**Expected output:**
```
✅ Connection to Neo4j established successfully!
⚡ Neo4j version: 5.26.x
```

In [ ]:
import time
from neo4j import GraphDatabase

# Connection URI using Bolt protocol (binary, high performance)
uri = "bolt://localhost:7688"

max_retries = 3
for attempt in range(1, max_retries + 1):
    try:
        # Create the connection driver (no authentication in local container)
        driver = GraphDatabase.driver(uri, auth=None, connection_timeout=10, max_connection_lifetime=60)
        
        # Verify connection by running a quick query
        with driver.session() as session:
            resultado = session.run(
                "CALL dbms.components() YIELD name, versions RETURN versions[0] AS versao"
            ).single()
            print("✅ Connection to Neo4j established successfully!")
            print(f"⚡ Neo4j version: {resultado['versao']}")
            break
    except Exception as e:
        print(f"⚠️ Attempt {attempt}/{max_retries} failed. Neo4j may still be initializing...")
        if attempt == max_retries:
            print(f"❌ Final error connecting to Neo4j: {e}")
            print("Make sure the Neo4j container is running and the ports are correctly mapped.")
        else:
            time.sleep(5)  # Wait 5 seconds before trying again


---
## 3. Cleaning the Test Database
Before creating new nodes and edges, let's run a Cypher command to remove all existing data, ensuring the notebook runs cleanly and reproducibly.

> **💡 Key Concept: Transactions.** In Neo4j, every operation occurs within a **transaction**. The `execute_write()` method ensures the operation runs as a write transaction — if an error occurs, changes are rolled back.

**Expected output:**
```
🧹 Graph database cleaned and ready!
```

In [ ]:
def limpar_banco(tx):
    """Remove all nodes and relationships from the database.
    DETACH DELETE first removes relationships, then the nodes."""
    tx.run("MATCH (n) DETACH DELETE n")

# execute_write() runs the function inside a write transaction
# The 'tx' (transaction) parameter is automatically injected by the driver
with driver.session() as session:
    session.execute_write(limpar_banco)
    print("🧹 Graph database cleaned and ready!")

---
## 4. CRUD — Create (Creating Nodes and Relationships)
We will use **parameterized** Cypher queries (with `$nome_variavel`). Parameterization is recommended to prevent injection attacks and improve Neo4j's execution plan caching.

Let's build the following social network:

```
    ┌─────────┐    AMIGO_DE     ┌─────────┐    AMIGO_DE    ┌─────────┐
    │  Alice  │ ──────────────► │   Bob   │ ─────────────► │ Charlie │
    └─────────┘                 └─────────┘                └─────────┘
         │
         │ AMIGO_DE
         ▼
    ┌─────────┐
    │  Diego  │
    └─────────┘
```

**Expected output:**
```
👤 Person nodes created successfully!
🔗 Friendship relationships created successfully!
```

In [ ]:
def criar_pessoa(tx, nome, idade, cidade):
    """Creates a node with label 'Pessoa' and the specified properties.
    CREATE always creates a new node (use MERGE to avoid duplicates)."""
    query = """
    CREATE (p:Pessoa {nome: $nome, idade: $idade, cidade: $cidade})
    RETURN p
    """
    # Parameters with $ are safely substituted by the driver
    tx.run(query, nome=nome, idade=idade, cidade=cidade)

def criar_amizade(tx, nome1, nome2, desde_ano):
    """Creates an AMIGO_DE relationship between two existing people.
    The relationship is DIRECTED (a -> b) and has the 'desde' property."""
    query = """
    MATCH (a:Pessoa {nome: $nome1})
    MATCH (b:Pessoa {nome: $nome2})
    CREATE (a)-[r:AMIGO_DE {desde: $desde_ano}]->(b)
    RETURN r
    """
    tx.run(query, nome1=nome1, nome2=nome2, desde_ano=desde_ano)

# Execute the creation operations within write transactions
with driver.session() as session:
    # 1. Create the Person Nodes
    session.execute_write(criar_pessoa, "Alice", 28, "João Pessoa")
    session.execute_write(criar_pessoa, "Bob", 30, "Recife")
    session.execute_write(criar_pessoa, "Charlie", 25, "Natal")
    session.execute_write(criar_pessoa, "Diego", 35, "João Pessoa")
    print("👤 Person nodes created successfully!")
    
    # 2. Create the Friendship Relationships (Directed edges)
    session.execute_write(criar_amizade, "Alice", "Bob", 2022)
    session.execute_write(criar_amizade, "Bob", "Charlie", 2023)
    session.execute_write(criar_amizade, "Alice", "Diego", 2021)
    print("🔗 Friendship relationships created successfully!")

---
## 5. CRUD — Read (Querying Data and Paths)
Let's run two queries that demonstrate the true power of graph databases:

1. **Find Alice's direct friends** (1-level traversal).
2. **Friend recommendation** — find "friends of friends" that Alice is not yet connected to (2-level traversal).

> **💡 Key Concept:** This type of query (graph traversal) is exactly where Neo4j excels compared to relational databases. In SQL, a "friend of a friend" recommendation would require multiple JOINs — in Cypher, it's a simple visual pattern.

**Expected output:**
```
📖 Alice's direct friends:
- Bob (30 years old), friends since 2022
- Diego (35 years old), friends since 2021
--------------------------------------------------
📊 Friend recommendations for Alice:
- Suggested: Charlie (Because they are friends of Bob)
```

In [ ]:
# === Query 1: Alice's direct friends ===
# The pattern (p)-[r:AMIGO_DE]->(amigo) reads as:
# "find person p who has an AMIGO_DE relationship pointing to friend"
query_amigos = """
MATCH (p:Pessoa {nome: $nome_alvo})-[r:AMIGO_DE]->(amigo)
RETURN amigo.nome AS nome, amigo.idade AS idade, r.desde AS ano_amizade
"""

with driver.session() as session:
    resultados = session.run(query_amigos, nome_alvo="Alice")
    print("📖 Alice's direct friends:")
    for registro in resultados:
        print(f"- {registro['nome']} ({registro['idade']} years old), friends since {registro['ano_amizade']}")

print("-" * 50)

# === Query 2: Recommendation (Alice's Friends of Friends) ===
# The two-hop pattern: Alice -> Friend -> FriendOfFriend
# WHERE ensures that:
#   1. We're not recommending someone Alice is already friends with
#   2. We're not recommending Alice herself
query_recomendacao = """
MATCH (alice:Pessoa {nome: 'Alice'})-[:AMIGO_DE]->(amigo)-[:AMIGO_DE]->(sugerido)
WHERE NOT (alice)-[:AMIGO_DE]->(sugerido) AND sugerido.nome <> 'Alice'
RETURN sugerido.nome AS recomendacao, amigo.nome AS intermediario
"""

with driver.session() as session:
    recomendacoes = session.run(query_recomendacao)
    print("📊 Friend recommendations for Alice:")
    for rec in recomendacoes:
        print(f"- Suggested: {rec['recomendacao']} (Because they are friends of {rec['intermediario']})")

---
## 6. CRUD — Update (Updating Nodes or Relationships)
Updates in Neo4j use the `SET` clause applied to nodes or relationships found via `MATCH`.

> **💡 Key Concept:** `MATCH` finds existing nodes/relationships, and `SET` modifies their properties. You can update both **nodes** and **relationships**.

**Expected output:**
```
🔄 Bob's age successfully updated to 31!
```

In [ ]:
# === Update Bob's age to 31 ===
# MATCH finds the node, SET modifies the property
query_update_idade = """
MATCH (p:Pessoa {nome: $nome})
SET p.idade = $nova_idade
RETURN p.nome AS nome, p.idade AS idade
"""

with driver.session() as session:
    resultado = session.run(query_update_idade, nome="Bob", nova_idade=31).single()
    print(f"🔄 {resultado['nome']}'s age successfully updated to {resultado['idade']}!")

---
## 7. CRUD — Delete (Deleting Nodes and Relationships)
In Neo4j, you **cannot delete a node** if it still has connected relationships (this ensures **referential integrity** of the graph — there can't be "dangling" edges pointing to nothing).

To work around this, we use `DETACH DELETE`, which:
1. First deletes **all** of the node's connections.
2. Then deletes the node itself.

**Expected output:**
```
🗑️ Charlie's node and all its friendship connections have been removed!

👥 Remaining people in the database:
- Alice
- Bob
- Diego
```

In [ ]:
# === Delete Charlie's node ===
# DETACH DELETE removes the node AND all its relationships
# Without DETACH, Neo4j would throw an error if the node has connections
query_delete = """
MATCH (p:Pessoa {nome: $nome_deletar})
DETACH DELETE p
"""

with driver.session() as session:
    session.run(query_delete, nome_deletar="Charlie")
    print("🗑️ Charlie's node and all its friendship connections have been removed!")

    # Show remaining people in the graph
    print("\n👥 Remaining people in the database:")
    todos = session.run("MATCH (p:Pessoa) RETURN p.nome AS nome")
    for pessoa in todos:
        print(f"- {pessoa['nome']}")

---
## 8. Closing the Connection
It is good practice to close the driver after use to release connections to the Neo4j server.

In [ ]:
# Close the Neo4j driver (ends all sessions and connections)
driver.close()
print("🔌 Connection to Neo4j closed successfully.")

---
## 🏁 Conclusion
Congratulations! You have completed the exercises with Neo4j and the Cypher language:
- ✅ Learned how to instantiate and manage connections via the binary **Bolt** protocol.
- ✅ Created dynamically structured **nodes** with Labels and properties.
- ✅ Created typed and directed **relationships** between entities in the graph.
- ✅ Understood how to navigate connections using Cypher patterns, including **recommendation** queries (friend of a friend).
- ✅ Updated node attributes and performed safe deletions using `DETACH DELETE`.

### 🚀 Next Steps
To further deepen your Neo4j knowledge, try:
1. **Shortest Path:** Find the shortest path between two nodes with `shortestPath()`.
2. **MERGE:** Create nodes/relationships only if they don't already exist (avoiding duplicates).
3. **Constraints and Indexes:** Ensure uniqueness and performance with `CREATE CONSTRAINT`.
4. **APOC Library:** Extension with hundreds of utility procedures for graph analysis.
5. **Graph Data Science:** Algorithms like PageRank, Community Detection, and Centrality.

### 📚 Useful References
- [Official Neo4j Documentation](https://neo4j.com/docs/)
- [Cypher Manual](https://neo4j.com/docs/cypher-manual/current/)
- [Neo4j Python Driver](https://neo4j.com/docs/python-manual/current/)
- [Neo4j Browser (Web Interface)](http://localhost:7475)
- [Neo4j Sandbox (free playground)](https://neo4j.com/sandbox/)